Concordance 02 — SpliceJAC
Standalone SpliceJAC run on the bone marrow CD34+ dataset.

Environment. scvelo + splicejac + pyvis + plotly. SpliceJAC downgrades anndata, so this MUST be a fresh env separate from the scjdo and dynamo notebooks:

conda create -n sjac python=3.10 -y && conda activate sjac
pip install scvelo spliceJAC pyvis plotly

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
SEED   = 0
N_PCS  = 30
TOP_K  = 30
RESULTS_DIR = Path('concordance_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)


## 1. Load + shared preprocessing

In [2]:
import scvelo as scv
import scanpy as sc
import numpy as np

adata = scv.datasets.bonemarrow()
scv.pp.filter_and_normalize(adata, min_shared_counts=20)
scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=30)
sc.tl.diffmap(adata, n_comps=15)

# Deterministic iroot — highest-CD34 cell in HSC_1
_label_col = next((c for c in ['clusters','celltype','cell_type','leiden']
                   if c in adata.obs), None)
_hsc_mask  = adata.obs[_label_col].astype(str).str.startswith('HSC').to_numpy() \
             if _label_col else np.ones(adata.n_obs, bool)
if 'CD34' in adata.var_names:
    _cd34 = adata[:, 'CD34'].X
    _cd34 = np.asarray(_cd34.todense()).ravel() if hasattr(_cd34, 'todense') \
            else np.asarray(_cd34).ravel()
else:
    _cd34 = np.zeros(adata.n_obs)
if _hsc_mask.any() and _cd34.max() > 0:
    _idx = np.flatnonzero(_hsc_mask)
    adata.uns['iroot'] = int(_idx[np.argmax(_cd34[_idx])])
else:
    adata.uns['iroot'] = int(np.argmax(_cd34)) if _cd34.max() > 0 else 0
sc.tl.dpt(adata)
adata.obs['pseudotime'] = adata.obs['dpt_pseudotime'].astype(np.float32)
pt = adata.obs['pseudotime'].to_numpy()
adata.obs['pseudotime'] = ((pt - np.nanmin(pt)) /
                            (np.nanmax(pt) - np.nanmin(pt) + 1e-9)).astype(np.float32)
adata.obs['cluster'] = adata.obs[_label_col].astype('category')
CLUSTERS   = adata.obs['cluster'].cat.categories.tolist()
GENE_NAMES = list(adata.var_names)
print(f'{adata.n_obs} cells × {adata.n_vars} genes  | '
      f'clusters: {CLUSTERS}  | iroot={adata.uns["iroot"]}')


Filtered out 7837 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Logarithmized X.
computing neighbors
    finished (0:00:10) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:01) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
5780 cells × 6482 genes  | clusters: ['HSC_1', 'HSC_2', 'Ery_1', 'Mono_1', 'Precursors', 'CLP', 'Mono_2', 'DCs', 'Ery_2', 'Mega']  | iroot=1722


In [3]:
import json
metadata = {
    'dataset':         'scvelo.datasets.bonemarrow (Setty 2019 CD34+)',
    'n_cells':         int(adata.n_obs),
    'n_genes':         int(adata.n_vars),
    'seed':            SEED,
    'top_k':           TOP_K,
    'n_pcs':           N_PCS,
    'cluster_order':   CLUSTERS,
    'iroot_cell_idx':  int(adata.uns['iroot']),
}
(RESULTS_DIR / 'shared_metadata.json').write_text(json.dumps(metadata, indent=2))
print('shared_metadata.json saved')


shared_metadata.json saved


## 2. SpliceJAC — per-cluster Jacobian from splicing kinetics

In [4]:
import splicejac as sjac
adata_sj = adata.copy()
adata_sj.obs['clusters'] = adata_sj.obs['cluster']

# splicejac needs n_top_genes <= smallest cluster size
_min_cluster = int(adata_sj.obs['clusters'].value_counts().min())
N_TOP_SJAC   = min(50, max(8, _min_cluster - 1))
print(f'[splicejac] smallest cluster has {_min_cluster} cells; '
      f'using n_top_genes={N_TOP_SJAC}')
sjac.tl.estimate_jacobian(adata_sj, n_top_genes=N_TOP_SJAC,
                          seed=SEED, nsim=10)

# average_jac[cluster] = [J (G×G concat spliced+unspliced),
#                        eigenvalues (G,) complex,
#                        eigenvectors (G,G) complex]
ajdict        = adata_sj.uns['average_jac']
sjac_hvg      = list(adata_sj.var_names[adata_sj.var['highly_variable']])
G_half        = len(sjac_hvg)
name_to_idx   = {g: i for i, g in enumerate(GENE_NAMES)}

inst_arr       = np.full(len(CLUSTERS), np.nan, dtype=np.float32)
gene_vec       = np.zeros((len(CLUSTERS), len(GENE_NAMES)), dtype=np.float32)
gene_inst_rank = np.zeros_like(gene_vec)
top_genes      = {}
for i, c in enumerate(CLUSTERS):
    if c not in ajdict: continue
    J_c, eigvals, eigvecs = ajdict[c]
    eigvals = np.asarray(eigvals); eigvecs = np.asarray(eigvecs)
    k = int(np.argmax(eigvals.real))
    inst_arr[i] = float(eigvals[k].real)
    v = eigvecs[:, k]
    v_per_gene = np.abs(v[:G_half]) + np.abs(v[G_half:G_half*2])
    v_per_gene = v_per_gene / (np.linalg.norm(v_per_gene) + 1e-12)
    # Embed in full gene namespace so it aligns with scjdo/dynamo
    v_full = np.zeros(len(GENE_NAMES), dtype=np.float32)
    for gname, mag in zip(sjac_hvg, v_per_gene):
        j = name_to_idx.get(gname)
        if j is not None: v_full[j] = float(mag)
    gene_vec[i]       = v_full / (np.linalg.norm(v_full) + 1e-12)
    gene_inst_rank[i] = np.abs(v_full)
    top_idx = np.argsort(v_full)[::-1]
    top_idx = [j for j in top_idx if v_full[j] > 0][:TOP_K]
    top_genes[c] = np.array([GENE_NAMES[j] for j in top_idx])
print(pd.Series({c: inst_arr[i] for i, c in enumerate(CLUSTERS)
                if not np.isnan(inst_arr[i])})
      .sort_values(ascending=False).round(3))

[splicejac] smallest cluster has 61 cells; using n_top_genes=50
Filtered out 387 genes that are detected 20 counts (shared).
Extracted 50 highly variable genes.
Running quick regression...
Running subset regression on the Mono_2 cluster...
Running subset regression on the Mono_1 cluster...
Running subset regression on the HSC_1 cluster...
Running subset regression on the HSC_2 cluster...
Running subset regression on the Ery_1 cluster...
Running subset regression on the Precursors cluster...
Running subset regression on the Mega cluster...
Running subset regression on the CLP cluster...
Running subset regression on the DCs cluster...
Running subset regression on the Ery_2 cluster...
Ery_2         0.032
Mono_1        0.017
HSC_2         0.015
Precursors    0.009
Mono_2        0.008
Ery_1         0.005
Mega          0.005
CLP           0.004
DCs           0.003
HSC_1         0.001
dtype: float32


## 3. Save standardized .npz

In [5]:
save_path = RESULTS_DIR / 'splicejac_per_cluster.npz'
np.savez_compressed(save_path,
    method='splicejac',
    clusters=np.array(CLUSTERS),
    inst_per_cluster=inst_arr,
    gene_names=np.array(GENE_NAMES),
    gene_vec=gene_vec,
    gene_inst_rank=gene_inst_rank,
    **{f'top_genes/{c}': top_genes[c] for c in top_genes})
print(f'Saved → {save_path}  ({save_path.stat().st_size:,} bytes)')

Saved → concordance_results/splicejac_per_cluster.npz  (52,741 bytes)
